# Weak-lensing galaxy shape catalogue validation

## Local metacalibration

Contents.
- Metacalibration (local)

### Spatial binning

In [ ]:
# Projection all objects from spherical to Cartesian coordinates
x, y = radec2xy(np.mean(dd['XWIN_WORLD']), np.mean(dd['YWIN_WORLD']), dd['XWIN_WORLD'], dd['YWIN_WORLD'])

#### Compute and enlarge field size

In [ ]:
# Define mix, max and size
min_x = np.min(x)
max_x = np.max(x)
min_y = np.min(y)
max_y = np.max(y)

size_x = max_x - min_x
size_y = max_y - min_y
size_x_deg = np.rad2deg(size_x)
size_y_deg = np.rad2deg(size_y)

print(f'Field size in projected coordinates is (x, y) = ({size_x_deg:.2f}, {size_y_deg:.2f}) deg')

In [ ]:
# Calibration pixel sizes for local calibration, in degree
cal_pix_size_deg = [4, 2, 1, 0.5]

In [ ]:
# Compute larger field size, divisible by largest subfield size
factor = max(subfield)

size_x_deg_new = np.ceil(size_x_deg / factor) * factor
size_y_deg_new = np.ceil(size_y_deg / factor) * factor

print(f'Enlarged field size in projected coordinates is (x, y) = ({size_x_deg_new:.1f}, {size_y_deg_new:.1f}) deg')

In [ ]:
# Modify max coordinates to account for enlarged field
max_x_new = max_x + np.deg2rad(size_x_deg_new - size_x_deg)
max_y_new = max_y + np.deg2rad(size_y_deg_new - size_y_deg)
size_x_new_check = max_x_new - min_x
size_y_new_check = max_y_new - min_y

# Check that new size is ok
print('Check new field size:', np.rad2deg(size_x_new_check), np.rad2deg(size_y_new_check))

#### Local calibration

In [ ]:
def local_calib(npix_x, npix_y, dm, sigm):
    g1_final = np.array([])
    g2_final = np.array([])
    g1_final_dm = np.array([])
    g2_final_dm = np.array([])
    R_selec = np.zeros((2, 2, npix_y, npix_x))
    R_shear = np.zeros((2, 2, npix_y, npix_x))
    ra_ngmix2 = np.array([])
    dec_ngmix2 = np.array([])
    R_selec_std = np.zeros((2, 2, npix_y, npix_x))
    R_shear_std = np.zeros((2, 2, npix_y, npix_x))
    c1_ngmix_cut = np.zeros((npix_y, npix_x))
    c2_ngmix_cut = np.zeros((npix_y, npix_x))
    err_c1_ngmix_cut = np.zeros((npix_y, npix_x))
    err_c2_ngmix_cut = np.zeros((npix_y, npix_x))
    
    ngal = bin2d(xx, yy, npix=(npix_x, npix_y), extent=(min_x, max_x, min_y, max_y))

    # Cut the catalogue into subpatches -> dd_cut
    for i in range(npix_x):
        for j in range(npix_y):
            if(i==npix_x and j!=npix_y):
                dd_cut = dd[np.where((x >= min_x + (i*size_x/npix_x)) & (x <= max_x) \
                                     & (y >= min_y + (j*size_y/npix_y)) & (y < min_y + ((j+1)*size_y/npix_y)))]
            elif(j==npix_x and i!=npix_y):
                dd_cut = dd[np.where((x >= min_x + (i*size_x/npix_x)) & (x < min_x + ((i+1)*size_x/npix_x)) \
                                     & (y >= min_y + (j*size_y/npix_y)) & (y <= max_y))] 
            elif(j==npix_x and i==npix_y):
                dd_cut = dd[np.where((x >= min_x + (i*size_x/npix_x)) & (x <= max_x) \
                                     & (y >= min_y + (j*size_y/npix_y)) & (y <= max_y ))]
            else:
                dd_cut = dd[np.where((x >= min_x + (i*size_x/npix_x)) & (x < min_x + ((i+1)*size_x/npix_x)) \
                                     & (y >= min_y + (j*size_y/npix_y)) & (y < min_y + ((j+1)*size_y/npix_y)))]
        
            # Mask of dd_cut
            sm_classif_cut = dd_cut['SPREAD_MODEL']+2*dd_cut['SPREADERR_MODEL']
            m_gal_ngmix_cut = \
                (sm_classif_cut > 0.0035) \
                & (dd_cut['SPREAD_MODEL'] > 0.) \
                & (dd_cut['SPREAD_MODEL'] < 0.03) \
                & (dd_cut['MAG_AUTO'] < 26) \
                & (dd_cut['MAG_AUTO'] > 20) \
                & (dd_cut['FLAGS'] == 0) \
                & (dd_cut['IMAFLAGS_ISO'] == 0) \
                & (dd_cut['NGMIX_MCAL_FLAGS'] == 0) \
                & (dd_cut['NGMIX_ELL_PSFo_NOSHEAR'][:,0] != -10) \
                & (dd_cut['NGMIX_MOM_FAIL'] == 0) \
                & (dd_cut['N_EPOCH'] > 0) \
                & (dd_cut['NGMIX_N_EPOCH'] > 0)
            print('Objects selected as galaxies = {}'.format(len(np.where(m_gal_ngmix_cut)[0])))

            # Apply metacal to dd_cut
            gal_metacal_ngmix_cut = metacal(dd_cut, m_gal_ngmix_cut)
        
            # Save ra & dec to keep the order        
            ra_ngmix_temp = dd_cut['XWIN_WORLD'][m_gal_ngmix_cut][gal_metacal_ngmix_cut.mask_dict['ns']]
            ra_ngmix2 = np.concatenate((ra_ngmix2,ra_ngmix_temp))
            dec_ngmix_temp = dd_cut['YWIN_WORLD'][m_gal_ngmix_cut][gal_metacal_ngmix_cut.mask_dict['ns']]
            dec_ngmix2 = np.concatenate((dec_ngmix2,dec_ngmix_temp))
            
            w_ngmix_cut = gal_metacal_ngmix_cut.ns['w'][gal_metacal_ngmix_cut.mask_dict['ns']]
        
            g_ngmix_cut = np.array([gal_metacal_ngmix_cut.ns['g1'][gal_metacal_ngmix_cut.mask_dict['ns']], gal_metacal_ngmix_cut.ns['g2'][gal_metacal_ngmix_cut.mask_dict['ns']]])
        
            # If numver of galaxy is ok, metacal local
            if (ngal[j,i] > (np.mean(ngal))/2):
                g_corr_ngmix_cut = np.linalg.inv(gal_metacal_ngmix_cut.R).dot(g_ngmix_cut)
                
                # Add of delta m 
                R_dm = gal_metacal_ngmix_cut.R + np.ones((2,2))*(dm + np.random.normal(0, sigm))
                g_corr_ngmix_cut_dm = np.linalg.inv(R_dm).dot(g_ngmix_cut)
                
                # Additive bias
                c1_ngmix_cut[j,i], err_c1_ngmix_cut[j,i] = jackknif_weighted_average2(g_corr_ngmix_cut[0], w_ngmix_cut, remove_size=0.05, n_realization=500)
                c2_ngmix_cut[j,i], err_c2_ngmix_cut[j,i] = jackknif_weighted_average2(g_corr_ngmix_cut[1], w_ngmix_cut, remove_size=0.05, n_realization=500)
                
                # Save R matrix and std
                R_selec[:,:,j,i] = gal_metacal_ngmix_cut.R_selection
                R_shear[:,:,j,i] = np.mean(gal_metacal_ngmix_cut.R_shear,2)
                R_selec_std[:,:,j,i] = gal_metacal_ngmix_cut.R_selection_std
                R_shear_std[:,:,j,i] = gal_metacal_ngmix_cut.R_shear_std
                                
            # If low number of galaxy, we use value of globaal metacal
            else:
                g_corr_ngmix_cut = np.linalg.inv(R_tot_moy).dot(g_ngmix_cut)
                R_dm = R_tot_moy + np.ones((2,2))*(dm + np.random.normal(0, sigm))
                g_corr_ngmix_cut_dm = np.linalg.inv(R_dm).dot(g_ngmix_cut)
                
                c1_ngmix_cut[j,i] = c1_ngmix
                err_c1_ngmix_cut[j,i] = err_c1_ngmix
                c2_ngmix_cut[j,i] = c2_ngmix
                err_c2_ngmix_cut[j,i] = err_c2_ngmix
            
                R_selec[:,:,j,i] = R_selec_moy
                R_shear[:,:,j,i] = R_shear_moy
                R_selec_std[:,:,j,i] = R_selec_moy_std
                R_shear_std[:,:,j,i] = R_shear_moy_std
                             
            g1_final = np.concatenate((g1_final, g_corr_ngmix_cut[0]))  
            g2_final = np.concatenate((g2_final, g_corr_ngmix_cut[1]))
            g1_final_dm = np.concatenate((g1_final_dm, g_corr_ngmix_cut_dm[0]))  
            g2_final_dm = np.concatenate((g2_final_dm, g_corr_ngmix_cut_dm[1]))
    
    #return g1_final, g2_final, g1_final_dm, g2_final_dm, R_selec, R_shear, ra_ngmix2, dec_ngmix2, R_selec_std, R_shear_std,c1_ngmix_cut,c2_ngmix_cut,err_c1_ngmix_cut,err_c2_ngmix_cut
    return (
        np.array([g1_final, g2_final]), R_shear, R_selection, ra_ngmix2, dec_ngmix2, R_shear_std, R_selec_std,
        np.array([c1_ngmix_cut, c2_ngmix_cut]), np.array([err_c1_ngmix_cut, err_c2_ngmix_cut])
    )

In [ ]:
# Project selected galaxies from spherical to Cartesian coordinates
xx, yy = radec2xy(np.mean(ra_ngmix), np.mean(dec_ngmix), ra_ngmix, dec_ngmix)

In [ ]:
g_corr_ngmix_local = {}
g_corr_ngmix_local = {}
R_shear_local = {}
R_selec_local = {}
ra_ngmix_local = {}
dec_ngmix_local = {}
R_shear_std_local = {}
R_selec_std_local = {}
c_ngmix_local = {}
c_err_ngmix_local = {}

# Additional mean and std of multiplicative bias
m = 0
dm = 0

# Loop over different calibration pixel sizes
for cal_pix in cal_pix_size_deg:
    npix_x = int(size_x_deg_new / cal_pix)
    npix_y = int(size_y_deg_new / cal_pix)
    print(cal_pix, npix_x, npix_y)
    
    (
        g_corr_ngmix_corr[cal_pix],
        R_shear_local[cal_pix],
        R_selection_local[cal_pix],
        ra_ngmix_local[cal_pix],
        dec_ngmix_local[cal_pix],
        R_shear_std_local[cal_pix],
        R_selection_std_local[cal_pix],
        c_ngmix_local[cal_pix],
        c_err_ngmix_local[cal_pix]
    ) = local_calib(npix_x, npix_y, m, dm)    